<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v3/mnps_new_baseline%20v7.5.3C1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity New Baseline 7.5.3C1**
> A notebook built on improvements  
> DSI DSSG + MNPS   

> # **Version 7.5.3C1 Changes**
> - **CRITICAL FIX**: Teacher/Librarian/Counselor/Principal/Assistant Principal now correctly have NO minor_sub_group (except "Lead")
> - **NEW ROLES**: Added Assistant, Representative, Therapist, and Instructor to MAJOR_ALLOWED
> - **Instructor vs Teacher**: Added logic to distinguish between Instructor (no teaching license required) and Teacher (requires teaching license)
> - **Representative Role**: Added Representative for roles like Facility Representative that don't meet Coordinator requirements
> - **Enhanced Role Classification**: Added Architect (Facility-Focused) role for better architectural role distinction
> - **Therapist Role**: Added distinct Therapist role (separate from Teacher)
> - **Assistant Role**: Added Assistant role for roles like Physical Therapist Assistant
> - **Director Minor Sub-Group**: Directors rarely have minor sub-grouping
> - **Accountant Level Logic**: Improved logic to determine appropriate Accountant level
> - **All v7.5.3.1 Features Retained**: Rate limiting, Problem Role Cheat Sheet logic, etc.

In [ ]:
# ==== 1) Imports, paths, inputs from v7.1 artifacts ====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")

In [ ]:
# ==== 2) Load data and build attribute-only view (ignore title) ====

# Load prediction data (if available from previous runs)
# For now, we'll work with the raw job descriptions
preds = df.copy()

# Build attribute-only text (ignore job titles)
ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]

# Combine all attribute text
attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']

print(f"✅ Built attribute-only view for {len(text)} job descriptions")
print(f"✅ Ignoring job titles - focusing on job attributes only")

In [ ]:
# ==== 3) Enhanced closed sets and normalization helpers - v7.5.3.C1 ====

# Get MNPS roles from the loaded data
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
if role_columns:
    VALID_ROLES = roles_df[role_columns[0]].dropna().tolist()
else:
    VALID_ROLES = roles_df.iloc[:, 0].dropna().tolist()

print(f"✅ Found {len(VALID_ROLES)} MNPS roles")

# EXPANDED closed sets for major and minor role groups - v7.5.3.C1
MAJOR_ALLOWED = [
    'Technician', 'Specialist', 'Analyst', 'Manager', 'Coordinator', 'Director', 'Other',
    'Teacher', 'Coach', 'Counselor', 'Clerical Support', 'Instructor', 'Driver',
    'Supervisor', 'Accountant', 'Architect (Facility-Focused)', 'Architect (Technology-Focused)',
    'Principal', 'Assistant Principal', 'Librarian', 'Social Worker', 'Therapist', 'Translator',
    'Skilled Laborer', 'Administrative Assistant', 'Assistant', 'Representative'
]

MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']

# Roles that should NOT have minor sub-grouping (except Lead)
NO_MINOR_ROLES = ['Teacher', 'Librarian', 'Counselor', 'Principal', 'Assistant Principal', 'Therapist']

# Executive roles that rarely have "Lead" minor sub-grouping
EXECUTIVE_ROLES = ['Coordinator', 'Director', 'Manager']

# Normalization mapping for minor roles
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

def normalize_minor(x: str) -> str:
    """Normalize minor role to approved values."""
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def extract_role_from_justification(justification: str) -> str:
    """CRITICAL: Extract the actual role mentioned in the justification.

    This is the key function that was causing mismatches.
    The justification often mentions the correct role, but the major_role_group field was wrong.
    """
    if not justification:
        return None

    just_text = str(justification)

    # Look for exact role mentions with proper priority order
    # Check most specific roles first to avoid false matches
    role_patterns = [
        # Multi-word roles first (more specific)
        (r'\b(Assistant Principal)\b', 'Assistant Principal'),
        (r'\b(Architect \(Facility-Focused\))\b', 'Architect (Facility-Focused)'),
        (r'\b(Architect \(Technology-Focused\))\b', 'Architect (Technology-Focused)'),
        (r'\b(Administrative Assistant)\b', 'Administrative Assistant'),
        (r'\b(Clerical Support)\b', 'Clerical Support'),
        (r'\b(Social Worker)\b', 'Social Worker'),
        (r'\b(Skilled Laborer)\b', 'Skilled Laborer'),

        # Single-word roles
        (r'\b(Representative)\b', 'Representative'),
        (r'\b(Coordinator)\b', 'Coordinator'),
        (r'\b(Principal)\b', 'Principal'),
        (r'\b(Director)\b', 'Director'),
        (r'\b(Manager)\b', 'Manager'),
        (r'\b(Supervisor)\b', 'Supervisor'),
        (r'\b(Analyst)\b', 'Analyst'),
        (r'\b(Accountant)\b', 'Accountant'),
        (r'\b(Technician)\b', 'Technician'),
        (r'\b(Therapist)\b', 'Therapist'),
        (r'\b(Counselor)\b', 'Counselor'),
        (r'\b(Librarian)\b', 'Librarian'),
        (r'\b(Translator)\b', 'Translator'),
        (r'\b(Instructor)\b', 'Instructor'),
        (r'\b(Teacher)\b', 'Teacher'),
        (r'\b(Coach)\b', 'Coach'),
        (r'\b(Driver)\b', 'Driver'),
        (r'\b(Assistant)\b', 'Assistant'),
        (r'\b(Specialist)\b', 'Specialist'),
    ]

    for pattern, role_name in role_patterns:
        if re.search(pattern, just_text, re.IGNORECASE):
            return role_name

    return None

def fix_no_minor_roles(major_role: str, minor_role: str) -> str:
    """CRITICAL: Teacher/Librarian/Counselor/Principal/Assistant Principal/Therapist should have NO minor sub-grouping (except Lead)."""
    if major_role in NO_MINOR_ROLES:
        if minor_role == 'Lead':
            return 'Lead'
        else:
            return ''  # No minor sub-grouping
    return minor_role

def fix_director_minor(major_role: str, minor_role: str) -> str:
    """Directors rarely have minor sub-grouping."""
    if major_role == 'Director' and minor_role in ['I', 'II', 'III']:
        return ''  # No minor sub-grouping for Directors
    return minor_role

def distinguish_instructor_teacher(row: pd.Series, proposed_major: str) -> str:
    """Distinguish between Instructor and Teacher based on teaching license requirements.

    Teacher: Requires teaching license/certification
    Instructor: Does not require teaching license (e.g., JROTC, vocational instructors)
    """
    if proposed_major not in ['Instructor', 'Teacher']:
        return proposed_major

    cert_text = str(row.get('Licenses and Certifications', '')).lower()
    func_text = str(row.get('Essential Functions', '')).lower()

    # Check for teaching license requirements
    has_teaching_license = re.search(r'(teaching license|teaching certification|teacher license|teacher certification|certificated)', cert_text)

    # Check for JROTC or other instructor-specific patterns
    is_jrotc = re.search(r'(jrotc|junior rotc|army instructor|military)', cert_text + func_text)

    if is_jrotc or (not has_teaching_license and 'instructor' in func_text):
        return 'Instructor'
    elif has_teaching_license or 'classroom' in func_text:
        return 'Teacher'

    return proposed_major

def determine_accountant_level(row: pd.Series) -> str:
    """Determine appropriate Accountant level based on experience and education."""
    exp_text = str(row.get('Work Experience', '')).lower()
    ed_text = str(row.get('Education', '')).lower()

    # Check for experience level
    has_extensive_exp = re.search(r'(5\+|five or more|5 or more|six|seven|eight|nine|ten) years', exp_text)
    has_some_exp = re.search(r'(3|4|three|four) years', exp_text)
    has_minimal_exp = re.search(r'(1|2|one|two) years', exp_text)

    # Check for education level
    has_bachelors = re.search(r'bachelor', ed_text)
    has_associates = re.search(r'associate', ed_text)

    # Determine level
    if has_extensive_exp and has_bachelors:
        return 'II'
    elif (has_minimal_exp or has_some_exp) and (has_associates or has_bachelors):
        return ''
    else:
        return 'II'

print("✅ v7.5.3.C1: Enhanced role extraction and normalization helpers defined")

In [ ]:
# ==== 4) Build comprehensive KSACs text from all MNPS resources ====

def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n\n"

    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()

    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)

    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n\n"

    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)

    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"

    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)

    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"

    return ksacs_text

KSACS_TEXT = build_ksacs_text()

print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")

In [ ]:
# ==== 5) Enhanced Zero Shot Prompt - v7.5.3.C1 ====
zero_shot_prompt = \
"""Objective: Evaluate and group jobs based on similarities in job functions, not job titles.

Process:
- Compare all jobs using: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary
- Group jobs with similar functions, responsibilities, and requirements, regardless of job titles
- Use MNPS standards and classifications for alignment
- Focus on actual work performed, not job titles
- Provide clear justification based on job attributes and MNPS standards
- **CRITICAL**: State the role classification explicitly in your justification (e.g., "This position aligns with the Coordinator role because...")

IMPORTANT ROLE DISTINCTIONS:

**Technician vs Specialist vs Analyst:**
- Technician: Hands-on technical work, equipment maintenance, repair, installation
- Analyst: Data analysis, research, evaluation, statistical work, reporting
- Specialist: Specialized knowledge in specific domain (use sparingly, prefer more specific roles)

**Coordinator vs Coach vs Manager:**
- Coordinator: Coordination, organization, facilitation, liaison work, program coordination
- Coach: Instructional support, mentoring, professional development, co-teaching
- Manager: Strategic planning, policy development, budget oversight, supervision

**Instructor vs Teacher:**
- Instructor: Does NOT require teaching license (e.g., JROTC, vocational instructors)
- Teacher: Requires teaching license/certification, classroom instruction

**Assistant vs Therapist:**
- Assistant: Supports licensed professionals (e.g., Physical Therapist Assistant)
- Therapist: Licensed professional providing therapy services

**Representative vs Coordinator:**
- Representative: High school education, basic liaison/representative duties
- Coordinator: College degree, program coordination, strategic facilitation

**Supervisor vs Manager:**
- Supervisor: Primarily manages people, no post-high school education required
- Manager: Strategic work beyond people management, requires minimum associates degree

**Skilled Laborer vs Technician:**
- Skilled Laborer: Trades work (plumbing, electrical, carpentry, HVAC)
- Technician: Technical equipment maintenance and troubleshooting

MINOR SUB-GROUP GUIDELINES:
- **NO minor sub-grouping** for: Teacher, Librarian, Counselor, Principal, Assistant Principal, Therapist (except "Lead")
- **Rarely minor sub-grouping** for: Director (usually blank)
- Executive roles (Coordinator, Manager) rarely use "Lead" - prefer I, II, or III
- III: Very advanced KSACs and senior-level expertise
- II: Intermediate complexity and responsibility
- I or blank: Entry-level or basic complexity
- Lead: Reserved for non-executive roles leading teams/projects

**CRITICAL INSTRUCTION**: In your grouping_justification, you MUST explicitly state the major role group you are selecting (e.g., "This position aligns with the Coordinator role...", "This role is best classified as an Analyst...", "This is a Therapist position because..."). This explicit statement is required for accurate classification.

Output Requirements:
- major_role_group: Choose from approved MNPS roles
- minor_sub_group: Use I, II, III, Lead, or blank based on guidelines above
- new_job_title: Incorporate both major_role_group and minor_sub_group (e.g., "Transportation Manager II", "Teacher", "Accountant")
- grouping_justification: MUST explicitly state the major role classification and explain why based on job attributes
"""

print("✅ v7.5.3.C1: Enhanced zero shot prompt defined")

In [ ]:
# ==== 6) OpenAI API Setup with Rate Limiting Protection ====
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()
MODEL_ID = "gpt-4o-2024-11-20"

print(f"✅ OpenAI client initialized")
print(f"✅ Using model: {MODEL_ID}")

def call_llm_json_with_retry(prompt: str, model: str = None, max_retries: int = 3) -> dict:
    """Call OpenAI API with JSON response and exponential backoff for rate limiting."""
    if model is None:
        model = MODEL_ID

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)

        except Exception as e:
            error_str = str(e).lower()

            if "429" in error_str or "rate limit" in error_str or "quota" in error_str:
                if attempt < max_retries - 1:
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"⚠️  Rate limit hit, waiting {wait_time:.1f} seconds before retry {attempt + 1}/{max_retries}")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Max retries reached for rate limiting. Error: {e}")
                    raise e
            else:
                print(f"❌ Non-rate limiting error: {e}")
                raise e

    raise Exception("Unexpected error in retry logic")

print("✅ call_llm_json_with_retry function defined")

In [ ]:
# ==== 7) Enhanced Batch Processing - v7.5.3.C1 with CRITICAL FIXES ====
from tqdm import tqdm

def process_job_description(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description using GPT-4o with v7.5.3.C1 critical fixes."""
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""

    prompt = f"""{zero_shot_prompt}

Available MNPS Roles: {', '.join(MAJOR_ALLOWED)}

{KSACS_TEXT}

Job Description to Classify:
{job_text}

**CRITICAL REQUIREMENTS**:
- Ignore job title completely
- Base classification solely on job attributes
- In your grouping_justification, explicitly state the major role (e.g., "This is a Coordinator role because...")
- Follow NO minor sub-grouping rule for Teacher/Librarian/Counselor/Principal/Assistant Principal/Therapist
- Distinguish Instructor (no teaching license) from Teacher (requires teaching license)
- Use Representative for roles with only high school education doing liaison work
- Use Skilled Laborer for trades work (plumbing, electrical, carpentry)
- Use Assistant for roles supporting licensed professionals
- Use Therapist for licensed therapy professionals

Return JSON:
{{
  "new_job_title": "Title with major_role_group and minor_sub_group",
  "major_role_group": "One of the approved MNPS roles",
  "minor_sub_group": "I, II, III, Lead, or blank",
  "grouping_justification": "MUST explicitly state the major role and explain why"
}}"""

    try:
        response_data = call_llm_json_with_retry(prompt, MODEL_ID)

        # Get initial values
        major_role = response_data.get('major_role_group', 'Other')
        minor_role = response_data.get('minor_sub_group', 'I')
        justification = response_data.get('grouping_justification', '') or ''
        new_job_title = response_data.get('new_job_title', 'Unknown')

        # CRITICAL FIX: Extract role from justification if present
        extracted_role = extract_role_from_justification(justification)
        if extracted_role and extracted_role in MAJOR_ALLOWED:
            major_role = extracted_role

        # Apply Instructor vs Teacher logic
        major_role = distinguish_instructor_teacher(row, major_role)

        # Normalize minor role
        if minor_role and str(minor_role).strip():
            minor_role = normalize_minor(minor_role)
        else:
            minor_role = ''

        # CRITICAL FIX: Apply NO minor sub-grouping rule
        minor_role = fix_no_minor_roles(major_role, minor_role)

        # Apply Director minor fix
        minor_role = fix_director_minor(major_role, minor_role)

        # Special handling for Accountant level
        if major_role == 'Accountant' and minor_role:
            accountant_level = determine_accountant_level(row)
            if accountant_level:
                minor_role = accountant_level
            else:
                minor_role = ''

        # Generate job title
        if not new_job_title or new_job_title == 'Unknown':
            if minor_role:
                new_job_title = f"{major_role} {minor_role}"
            else:
                new_job_title = major_role

        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': new_job_title.strip(),
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'grouping_justification': justification or 'No justification provided',
            'model_used': MODEL_ID
        }
    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all job descriptions
results = []
print("🚀 Starting v7.5.3.C1 batch processing...")

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    time.sleep(0.2)

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v753c1.csv"
results_df.to_csv(output_path, index=False)

print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")

In [ ]:
# ==== 8) Generate Summary Statistics and Examples ====

preds = results_df.copy()

major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()

summary_stats = pd.DataFrame({
    'metric': ['total_rows', 'unique_major_roles', 'unique_minor_roles', 'specialist_count',
               'no_minor_count', 'executive_lead_count'],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group'] == 'Specialist').sum()),
        int((preds['minor_sub_group'] == '').sum()),
        int((preds['major_role_group'].isin(EXECUTIVE_ROLES) & (preds['minor_sub_group'] == 'Lead')).sum())
    ]
})

summary_path = OUTPUTS_DIR / "summary_stats_gpt4o_v753c1.csv"
summary_stats.to_csv(summary_path, index=False)

examples = preds[['source_row_index', 'job_title_original', 'new_job_title',
                  'major_role_group', 'minor_sub_group']].head(15)

examples_path = OUTPUTS_DIR / "examples_gpt4o_v753c1.csv"
examples.to_csv(examples_path, index=False)

print("\n📊 v7.5.3.C1 Summary Statistics:")
print(summary_stats.to_string(index=False))

print("\n📊 Major Role Distribution:")
print(major_counts.to_string())

print("\n📊 Minor Role Distribution:")
print(minor_counts.to_string())

print("\n📋 Example Classifications:")
print(examples.to_string(index=False))

print(f"\n✅ Saved summary to: {summary_path}")
print(f"✅ Saved examples to: {examples_path}")